# Fetcher Pipeline

Notebook wrapper around the raw UN Comtrade extraction stage.

This notebook preserves the original `.py` scripts and runs them in place.


## How To Use

1. Open this notebook from the repo root or from the `notebooks/` folder.
2. Run the setup cell once.
3. Run the script cells you need. They execute the existing source files with the correct working directory.


In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import runpy
import sys


ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()


def run_project_script(relative_path: str, argv: list[str] | None = None) -> None:
    script_path = (ROOT / relative_path).resolve()
    script_dir = script_path.parent
    old_cwd = Path.cwd()
    old_argv = sys.argv[:]
    try:
        os.chdir(script_dir)
        sys.argv = [str(script_path), *(argv or [])]
        runpy.run_path(str(script_path), run_name="__main__")
    finally:
        os.chdir(old_cwd)
        sys.argv = old_argv


def show_stage_files(relative_dir: str, pattern: str = "*") -> list[Path]:
    base = (ROOT / relative_dir).resolve()
    files = sorted(base.glob(pattern))
    for path in files:
        print(path.relative_to(ROOT))
    return files


In [ ]:
print("Stage directory: 0fetcher")
show_stage_files("0fetcher", "*.py")


## Notes

- The fetch step calls the live UN Comtrade API and writes CSV files into `0fetcher/data_raw`.
- The fetcher script currently contains a hard-coded subscription key. The notebook does not modify that behavior.

## `0fetcher/0fetcher.py`

Download raw export and import CSV files for the configured commodities and years.


In [ ]:
run_project_script("0fetcher/0fetcher.py", argv=[])


## `0fetcher/1compile_to_excel.py`

Combine raw CSV files into a single multi-sheet Excel workbook.


In [ ]:
run_project_script("0fetcher/1compile_to_excel.py", argv=[])


## `0fetcher/2counter.py`

Count rows in each raw CSV file and write `row_counts.csv`.


In [ ]:
run_project_script("0fetcher/2counter.py", argv=[])
